In [ ]:
import pandas

In [ ]:
import pybaseball
from pybaseball import statcast

pybaseball.cache.enable()


This is a large query, it may take a moment to complete


c:\Users\ethro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pybaseball\statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
100%|██████████| 188/188 [00:44<00:00,  4.18it/s]


In [ ]:
sc = statcast(start_dt="2025-03-27", end_dt="2025-09-30")
sc = sc[[
    "game_date", "game_pk", "pitcher", "player_name", "batter",
    "pitch_type", "release_speed", "release_spin_rate",
    "description", "events", "inning", "outs_when_up",
    "home_team", "away_team", "stand", "p_throws"
]]

       game_date  game_pk  pitcher       player_name  batter pitch_type  \
549   2025-09-30   813074   547973  Chapman, Aroldis  663757         FF   
561   2025-09-30   813074   547973  Chapman, Aroldis  663757         SL   
576   2025-09-30   813074   547973  Chapman, Aroldis  663757         FF   
587   2025-09-30   813074   547973  Chapman, Aroldis  663757         FF   
604   2025-09-30   813074   547973  Chapman, Aroldis  663757         FF   
...          ...      ...      ...               ...     ...        ...   
3721  2025-03-27   778545   650633     King, Michael  663586         FF   
3845  2025-03-27   778545   650633     King, Michael  663586         FF   
3933  2025-03-27   778545   650633     King, Michael  595777         CH   
4099  2025-03-27   778545   650633     King, Michael  595777         SI   
4242  2025-03-27   778545   650633     King, Michael  595777         FF   

      release_speed  release_spin_rate      description     events  inning  \
549           101.2  

In [ ]:
sc["is_k"] = sc["events"].isin(["strikeout", "strikeout_double_play"]).astype(int)

# swinging strike flag
sc["is_whiff"] = sc["description"].isin([
    "swinging_strike",
    "swinging_strike_blocked"
]).astype(int)

pitcher_games = (
    sc.groupby(["game_date", "game_pk", "pitcher", "player_name"])
      .agg(
          pitches=("pitch_type", "count"),
          strikeouts=("is_k", "sum"),
          whiffs=("is_whiff", "sum"),
          avg_velo=("release_speed", "mean"),
          avg_spin=("release_spin_rate", "mean"),
          batters_faced=("batter", "nunique")
      )
      .reset_index()
)

pitcher_games["whiff_per_pitch"] = pitcher_games["whiffs"] / pitcher_games["pitches"]

In [ ]:
pitcher_games = pitcher_games.sort_values(["pitcher", "game_date"])

for col in ["strikeouts", "pitches", "whiff_per_pitch", "avg_velo", "avg_spin"]:
    pitcher_games[f"{col}_last3"] = (
        pitcher_games.groupby("pitcher")[col]
        .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    )
    pitcher_games[f"{col}_last10"] = (
        pitcher_games.groupby("pitcher")[col]
        .transform(lambda s: s.shift(1).rolling(10, min_periods=3).mean())
    )

In [ ]:
features = [
    "pitches_last3",
    "pitches_last10",
    "whiff_per_pitch_last3",
    "avg_velo_last3",
    "avg_spin_last3",
]
X = pitcher_games[features]

       pitches_last3  pitches_last10  whiff_per_pitch_last3  avg_velo_last3  \
345              NaN             NaN                    NaN             NaN   
864        83.000000             NaN               0.096386       88.704819   
1453       74.000000             NaN               0.140500       88.507794   
2155       79.333333       79.333333               0.160334       89.082233   
2678       86.333333       85.500000               0.153846       89.056909   
...              ...             ...                    ...             ...   
19394      34.000000       29.200000               0.056739       85.667857   
19458      26.000000       29.600000               0.101852       84.729762   
20045      22.666667       29.700000               0.126984       84.402646   
20449      20.333333       27.900000               0.131454       83.680232   
20788      21.333333       23.200000               0.118849       84.201660   

       avg_spin_last3  
345               NaN  
864

In [ ]:
model_df = pitcher_games[features + ["strikeouts"]].dropna()

X = model_df[features]
y = model_df["strikeouts"]

In [ ]:
import sklearn
import xgboost
print("sklearn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)

sklearn: 1.8.0
xgboost: 3.2.0


In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

ImportError: sklearn needs to be installed in order to use this module